# Seattle Permit Predictor — Random Forest Model (with Comment Features)
*5-Fold Cross-Validated Random Forest | Target: `log(1 + totaldaysplanreview)`*

**This notebook extends `PermitPredictor_RF.ipynb` by joining comment-derived features from `comment_features.csv` onto the master dataset before training.**

**Comment features added (9 selected from CommentFeatures.ipynb signal analysis):**
- `comment_max_cycle` — maximum review cycle number
- `comment_n_distinct_cycles` — number of distinct review cycles
- `comment_n_rows` — total correction comment rows
- `comment_n_subjects` — number of distinct correction subjects
- `comment_n_review_types` — number of distinct review types
- `comment_has_structural` — Structural Engineer review flag
- `comment_has_eca` — any ECA (Environmentally Critical Area) review flag
- `comment_has_design_review` — Design Review flag
- `comment_resubmit_rate` — fraction of comments containing resubmit/revise language

**Coverage:** Only ~5% of permits have comment data. All comment features are null for the remaining 95% and are median-imputed by the pipeline.

**Modeling population:** All 14,201 completed post-2019 permits with recorded review times (broad model — no dwelling unit type filter).

## 0. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import warnings
import datetime
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
OUTPUT_DIR   = r'C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output' + '\\'
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded.')

## 1. Load & Join Data

In [ ]:
# Load master dataset
df = pd.read_csv(f'{OUTPUT_DIR}master_dataset.csv')
print(f'Master dataset   : {len(df):,} rows')

# Load comment features produced by CommentFeatures.ipynb
comment_features = pd.read_csv(f'{OUTPUT_DIR}comment_features.csv')
print(f'Comment features : {len(comment_features):,} rows × {len(comment_features.columns)} columns')

# Left join — permits without comment data get nulls, imputed later
df = df.merge(comment_features, on='permitnum', how='left')

has_comments = df['comment_n_rows'].notna().sum()
print(f'\nAfter join: {len(df):,} rows')
print(f'  With comment features    : {has_comments:,} ({has_comments/len(df)*100:.1f}%)')
print(f'  Without (null-imputed)   : {len(df) - has_comments:,} ({(len(df)-has_comments)/len(df)*100:.1f}%)')
print(f'\nReview time stats (days):')
print(df['totaldaysplanreview'].describe().round(1))

## 2. Feature Selection & Preprocessing

The 9 comment features with strongest signal (selected from CommentFeatures.ipynb signal validation) are added to the numeric feature set. All are median-imputed to handle the 95% null rate for permits without comment data.

In [ ]:
CAT_FEATURES = ['permittypedesc', 'permitclass', 'zone_family']

BASE_NUM_FEATURES = [
    'log_estprojectcost',
    'log_housingunitsadded',
    'latitude',
    'longitude',
]

COMMENT_FEATURES = [
    'comment_max_cycle',
    'comment_n_distinct_cycles',
    'comment_n_rows',
    'comment_n_subjects',
    'comment_n_review_types',
    'comment_has_structural',
    'comment_has_eca',
    'comment_has_design_review',
    'comment_resubmit_rate',
]

NUM_FEATURES = BASE_NUM_FEATURES + COMMENT_FEATURES
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES
TARGET       = 'log_target'

for col in CAT_FEATURES:
    df[col] = df[col].fillna('Unknown')

X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()

print('Feature set:')
print(f'  Categorical (3)    : {CAT_FEATURES}')
print(f'  Base numeric (4)   : {BASE_NUM_FEATURES}')
print(f'  Comment numeric (9): {COMMENT_FEATURES}')
print(f'\nX shape: {X.shape} | y shape: {y.shape}')
print(f'\nNull counts per feature:')
print(X.isnull().sum())

In [ ]:
cat_pipe = Pipeline([
    ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median'))
])

preprocessor = ColumnTransformer([
    ('cat', cat_pipe, CAT_FEATURES),
    ('num', num_pipe, NUM_FEATURES)
], remainder='drop')

print('Preprocessor configured.')
print(f'  Categorical features : {len(CAT_FEATURES)}')
print(f'  Numeric features     : {len(NUM_FEATURES)} ({len(BASE_NUM_FEATURES)} base + {len(COMMENT_FEATURES)} comment)')

## 3. Baseline Random Forest — 5-Fold Cross Validation

In [ ]:
rf_baseline = RandomForestRegressor(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model_pipeline = Pipeline([
    ('prep', preprocessor),
    ('rf', rf_baseline)
])

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_pipeline.fit(X_train, y_train)
    y_pred_log = model_pipeline.predict(X_val)

    rmse_log = np.sqrt(mean_squared_error(y_val, y_pred_log))
    mae_log  = mean_absolute_error(y_val, y_pred_log)
    r2       = r2_score(y_val, y_pred_log)

    y_val_days  = np.expm1(y_val)
    y_pred_days = np.expm1(y_pred_log)
    rmse_days   = np.sqrt(mean_squared_error(y_val_days, y_pred_days))
    mae_days    = mean_absolute_error(y_val_days, y_pred_days)

    fold_results.append({
        'fold': fold,
        'rmse_log': rmse_log,
        'mae_log': mae_log,
        'r2': r2,
        'rmse_days': rmse_days,
        'mae_days': mae_days
    })
    print(f'Fold {fold} | R²={r2:.4f} | RMSE(log)={rmse_log:.4f} | MAE(days)={mae_days:.1f}')

results_df = pd.DataFrame(fold_results)
print('\n--- 5-Fold Cross Validation Summary (Baseline Random Forest) ---')
print(results_df.drop('fold', axis=1).describe().loc[['mean', 'std']].round(4))

## 4. Hyperparameter Tuning — Grid Search

In [ ]:
param_grid = {
    'rf__n_estimators': [200, 400],
    'rf__max_depth': [None, 20, 30],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__max_features': ['sqrt', 0.5]
}

grid_search = GridSearchCV(
    estimator=Pipeline([
        ('prep', preprocessor),
        ('rf', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    param_grid=param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X, y)

print('\nBest parameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nBest cross validation RMSE (log scale): {-grid_search.best_score_:.4f}')

## 5. Tuned Random Forest — 5-Fold Cross Validation

In [ ]:
best_params = {k.replace('rf__', ''): v for k, v in grid_search.best_params_.items()}

rf_tuned = RandomForestRegressor(
    **best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

tuned_pipeline = Pipeline([
    ('prep', preprocessor),
    ('rf', rf_tuned)
])

tuned_results = []
oof_preds = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    tuned_pipeline.fit(X_train, y_train)
    y_pred_log = tuned_pipeline.predict(X_val)
    oof_preds[val_idx] = y_pred_log

    rmse_log = np.sqrt(mean_squared_error(y_val, y_pred_log))
    mae_log  = mean_absolute_error(y_val, y_pred_log)
    r2       = r2_score(y_val, y_pred_log)

    y_val_days  = np.expm1(y_val)
    y_pred_days = np.expm1(y_pred_log)
    rmse_days   = np.sqrt(mean_squared_error(y_val_days, y_pred_days))
    mae_days    = mean_absolute_error(y_val_days, y_pred_days)

    tuned_results.append({
        'fold': fold,
        'rmse_log': rmse_log,
        'mae_log': mae_log,
        'r2': r2,
        'rmse_days': rmse_days,
        'mae_days': mae_days
    })
    print(f'Fold {fold} | R²={r2:.4f} | RMSE(log)={rmse_log:.4f} | MAE(days)={mae_days:.1f}')

tuned_df = pd.DataFrame(tuned_results)
print('\n--- 5-Fold Cross Validation Summary (Tuned Random Forest) ---')
print(tuned_df.drop('fold', axis=1).describe().loc[['mean', 'std']].round(4))

## 6. Evaluation Plots

In [ ]:
# --- Per-fold metrics bar chart ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = [('r2', 'R²', 'steelblue'), ('rmse_log', 'RMSE (log scale)', 'darkorange'), ('mae_days', 'MAE (days)', 'seagreen')]

for ax, (col, label, color) in zip(axes, metrics):
    ax.bar(tuned_df['fold'], tuned_df[col], color=color, alpha=0.8, edgecolor='white')
    mean_val = tuned_df[col].mean()
    ax.axhline(mean_val, color='black', linestyle='--', linewidth=1.2, label=f'Mean: {mean_val:.3f}')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Fold')
    ax.set_xticks(tuned_df['fold'])
    ax.legend(fontsize=9)

plt.suptitle('5-Fold Cross Validation Performance — Tuned Random Forest + Comment Features', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}cv_metrics_per_fold_comments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cv_metrics_per_fold_comments.png')

In [ ]:
# --- Out-of-fold predicted vs actual ---
actual_days = np.expm1(y.values)
pred_days   = np.expm1(oof_preds)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(actual_days, pred_days, alpha=0.2, s=10, color='steelblue', rasterized=True)
max_val = max(actual_days.max(), pred_days.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1.2, label='Perfect prediction')
ax.set_xlabel('Actual Review Days', fontsize=11)
ax.set_ylabel('Predicted Review Days', fontsize=11)
ax.set_title('Out-of-Fold Predicted vs Actual (days)\nBroad Model + Comment Features', fontsize=12, fontweight='bold')
ax.legend()

overall_r2  = r2_score(y.values, oof_preds)
overall_mae = mean_absolute_error(actual_days, pred_days)
ax.text(0.05, 0.92, f'OOF R² = {overall_r2:.3f}\nOOF MAE = {overall_mae:.0f} days',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}oof_pred_vs_actual_comments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: oof_pred_vs_actual_comments.png')

In [ ]:
# --- Residual analysis ---
residuals_log = y.values - oof_preds

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(residuals_log, bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.2)
axes[0].set_xlabel('Residual (log scale)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Residual Distribution (log scale)', fontsize=12, fontweight='bold')

axes[1].scatter(oof_preds, residuals_log, alpha=0.2, s=10, color='darkorange', rasterized=True)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Predicted (log scale)', fontsize=11)
axes[1].set_ylabel('Residual (log scale)', fontsize=11)
axes[1].set_title('Residuals vs Predicted', fontsize=12, fontweight='bold')

plt.suptitle('Out-of-Fold Residual Analysis — Broad Model + Comment Features', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}residual_analysis_comments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: residual_analysis_comments.png')

## 7. Feature Importance

Refit on full dataset. Comment features will appear alongside base features — their relative importance indicates how much signal they added despite 95% null coverage.

In [ ]:
tuned_pipeline.fit(X, y)

feature_names = ALL_FEATURES
importances   = tuned_pipeline.named_steps['rf'].feature_importances_

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df['type'] = imp_df['feature'].apply(
    lambda f: 'categorical' if f in CAT_FEATURES else ('comment' if f in COMMENT_FEATURES else 'base numeric')
)
imp_df = imp_df.sort_values('importance', ascending=True)

color_map = {'categorical': 'steelblue', 'base numeric': 'darkorange', 'comment': 'seagreen'}
colors = [color_map[t] for t in imp_df['type']]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(imp_df['feature'], imp_df['importance'], color=colors, alpha=0.85, edgecolor='white')

for bar, val in zip(bars, imp_df['importance']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=8)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue',  alpha=0.85, label='Categorical'),
    Patch(facecolor='darkorange', alpha=0.85, label='Base numeric'),
    Patch(facecolor='seagreen',   alpha=0.85, label='Comment feature'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.set_xlabel('Mean Decrease in Impurity', fontsize=11)
ax.set_title('Feature Importance — Broad Model + Comment Features\n(full dataset fit)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}feature_importance_comments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance_comments.png')
print()
print(imp_df.sort_values('importance', ascending=False)[['feature','type','importance']].to_string(index=False))

## 8. Sanity Check — Predicted vs Actual

In [ ]:
df_pred = df.copy()
for col in CAT_FEATURES:
    df_pred[col] = df_pred[col].fillna('Unknown')

df_pred['predicted_log']  = tuned_pipeline.predict(df_pred[ALL_FEATURES])
df_pred['predicted_days'] = np.expm1(df_pred['predicted_log'])
df_pred['actual_days']    = np.expm1(df_pred['log_target'])

print('=== Median Predicted vs Actual Days by Permit Type ===')
pt = df_pred.groupby('permittypedesc').agg(
    n=('actual_days', 'count'),
    actual_median=('actual_days', 'median'),
    predicted_median=('predicted_days', 'median')
).round(1).sort_values('actual_median', ascending=False)
print(pt.to_string())

print('\n=== Median Predicted vs Actual Days by Permit Class ===')
pc = df_pred.groupby('permitclass').agg(
    n=('actual_days', 'count'),
    actual_median=('actual_days', 'median'),
    predicted_median=('predicted_days', 'median')
).round(1).sort_values('actual_median', ascending=False)
print(pc.to_string())

## 9. Inference Function

In [ ]:
def predict_permit_review_time(
    permittypedesc: str,
    permitclass: str,
    zone_family: str,
    estprojectcost: float = None,
    housingunitsadded: float = None,
    latitude: float = None,
    longitude: float = None,
    # Comment features — all optional, median-imputed if not provided
    comment_max_cycle: float = None,
    comment_n_distinct_cycles: float = None,
    comment_n_rows: float = None,
    comment_n_subjects: float = None,
    comment_n_review_types: float = None,
    comment_has_structural: float = None,
    comment_has_eca: float = None,
    comment_has_design_review: float = None,
    comment_resubmit_rate: float = None,
    pipeline=None,
    n_trees_ci: bool = True
) -> dict:
    """
    Predict permit review time.
    Comment features are optional — if not provided, median imputation is applied.
    """
    if pipeline is None:
        pipeline = tuned_pipeline

    row = pd.DataFrame([{
        'permittypedesc':           permittypedesc,
        'permitclass':              permitclass,
        'zone_family':              zone_family,
        'log_estprojectcost':       np.log1p(estprojectcost)    if estprojectcost    is not None else np.nan,
        'log_housingunitsadded':    np.log1p(housingunitsadded) if housingunitsadded is not None else np.nan,
        'latitude':                 latitude   if latitude   is not None else np.nan,
        'longitude':                longitude  if longitude  is not None else np.nan,
        'comment_max_cycle':        comment_max_cycle        if comment_max_cycle        is not None else np.nan,
        'comment_n_distinct_cycles':comment_n_distinct_cycles if comment_n_distinct_cycles is not None else np.nan,
        'comment_n_rows':           comment_n_rows           if comment_n_rows           is not None else np.nan,
        'comment_n_subjects':       comment_n_subjects       if comment_n_subjects       is not None else np.nan,
        'comment_n_review_types':   comment_n_review_types   if comment_n_review_types   is not None else np.nan,
        'comment_has_structural':   comment_has_structural   if comment_has_structural   is not None else np.nan,
        'comment_has_eca':          comment_has_eca          if comment_has_eca          is not None else np.nan,
        'comment_has_design_review':comment_has_design_review if comment_has_design_review is not None else np.nan,
        'comment_resubmit_rate':    comment_resubmit_rate    if comment_resubmit_rate    is not None else np.nan,
    }])

    pred_log  = pipeline.predict(row)[0]
    pred_days = np.expm1(pred_log)

    result = {
        'predicted_days': round(pred_days, 1),
        'predicted_log':  round(pred_log, 4)
    }

    if n_trees_ci:
        prep       = pipeline.named_steps['prep']
        rf         = pipeline.named_steps['rf']
        X_t        = prep.transform(row)
        tree_preds = np.array([t.predict(X_t)[0] for t in rf.estimators_])
        result['ci_low_days']  = round(np.expm1(np.percentile(tree_preds, 10)), 1)
        result['ci_high_days'] = round(np.expm1(np.percentile(tree_preds, 90)), 1)

    return result


# --- Example predictions ---
# Without comment features (median-imputed)
examples = [
    {'permittypedesc': 'New',                 'permitclass': 'Single Family/Duplex', 'zone_family': 'SF',  'estprojectcost': 500_000,   'housingunitsadded': 1, 'latitude': 47.65, 'longitude': -122.35},
    {'permittypedesc': 'Addition/Alteration', 'permitclass': 'Single Family/Duplex', 'zone_family': 'NR',  'estprojectcost': 250_000,   'housingunitsadded': 1, 'latitude': 47.60, 'longitude': -122.32},
    {'permittypedesc': 'New',                 'permitclass': 'Multifamily',           'zone_family': 'LR',  'estprojectcost': 2_000_000, 'housingunitsadded': 8, 'latitude': 47.68, 'longitude': -122.34},
    {'permittypedesc': 'New',                 'permitclass': 'Commercial',            'zone_family': 'NC',  'estprojectcost': 1_500_000, 'housingunitsadded': 0, 'latitude': 47.61, 'longitude': -122.33},
]

print('=== Example Predictions (no comment features — median imputed) ===')
for ex in examples:
    result = predict_permit_review_time(**ex)
    label  = f"{ex['permittypedesc']} | {ex['permitclass']} | zone={ex['zone_family']}"
    print(f'\n  {label}')
    print(f"    Predicted : {result['predicted_days']:.0f} days")
    if 'ci_low_days' in result:
        print(f"    80% CI    : {result['ci_low_days']:.0f} – {result['ci_high_days']:.0f} days")

# With comment features supplied
print('\n=== Example Prediction (with comment features supplied) ===')
enriched = predict_permit_review_time(
    permittypedesc='New', permitclass='Single Family/Duplex', zone_family='SF',
    estprojectcost=500_000, housingunitsadded=1, latitude=47.65, longitude=-122.35,
    comment_max_cycle=3, comment_n_distinct_cycles=3, comment_n_rows=45,
    comment_n_subjects=18, comment_n_review_types=6,
    comment_has_structural=1, comment_has_eca=1, comment_has_design_review=0,
    comment_resubmit_rate=0.25
)
print(f"\n  New | Single Family/Duplex | zone=SF | $500k | with comment features")
print(f"    Predicted : {enriched['predicted_days']:.0f} days")
if 'ci_low_days' in enriched:
    print(f"    80% CI    : {enriched['ci_low_days']:.0f} – {enriched['ci_high_days']:.0f} days")

## 10. Save Model Weights

In [ ]:
model_save_path = f'{OUTPUT_DIR}ModelWeights_RF_Comments.joblib'
joblib.dump(tuned_pipeline, model_save_path)
print(f'Model saved: {model_save_path}')

## 11. Full Diagnostic Report

In [ ]:
sep  = '=' * 70
sep2 = '-' * 70
now  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

lines = []
lines.append(sep)
lines.append('  SEATTLE PERMIT PREDICTOR — RANDOM FOREST + COMMENT FEATURES REPORT')
lines.append(f'  Generated: {now}')
lines.append(sep)

# ── 1. Dataset ────────────────────────────────────────────────────────────
lines.append('')
lines.append('[ 1 ] DATASET')
lines.append(sep2)
lines.append(f'  Rows in modeling population : {len(df):,}')
lines.append(f'  With comment features       : {has_comments:,} ({has_comments/len(df)*100:.1f}%)')
lines.append(f'  Without (null-imputed)      : {len(df)-has_comments:,} ({(len(df)-has_comments)/len(df)*100:.1f}%)')
lines.append(f'  Categorical features ({len(CAT_FEATURES)})    : {CAT_FEATURES}')
lines.append(f'  Base numeric features ({len(BASE_NUM_FEATURES)})  : {BASE_NUM_FEATURES}')
lines.append(f'  Comment features ({len(COMMENT_FEATURES)})       : {COMMENT_FEATURES}')
lines.append(f'  Target                      : log(1 + totaldaysplanreview)')
lines.append('')
lines.append('  Null counts per feature:')
for col, n in X.isnull().sum().items():
    pct = n / len(X) * 100
    lines.append(f'    {col:<30} {n:>6,}  ({pct:.1f}%)')

# ── 2. Baseline Cross Validation ──────────────────────────────────────────
lines.append('')
lines.append('[ 2 ] BASELINE RANDOM FOREST — 5-FOLD CROSS VALIDATION')
lines.append(sep2)
lines.append('  n_estimators=200, all other params=default')
lines.append('')
lines.append(f"  {'Fold':>4}  {'R²':>7}  {'RMSE(log)':>10}  {'MAE(log)':>9}  {'RMSE(days)':>11}  {'MAE(days)':>10}")
lines.append('  ' + '-' * 60)
for _, row in results_df.iterrows():
    lines.append(f"  {int(row.fold):>4}  {row.r2:>7.4f}  {row.rmse_log:>10.4f}  {row.mae_log:>9.4f}  {row.rmse_days:>11.1f}  {row.mae_days:>10.1f}")
lines.append('  ' + '-' * 60)
m = results_df.drop('fold', axis=1).mean()
s = results_df.drop('fold', axis=1).std()
lines.append(f"  {'MEAN':>4}  {m.r2:>7.4f}  {m.rmse_log:>10.4f}  {m.mae_log:>9.4f}  {m.rmse_days:>11.1f}  {m.mae_days:>10.1f}")
lines.append(f"  {'STD':>4}  {s.r2:>7.4f}  {s.rmse_log:>10.4f}  {s.mae_log:>9.4f}  {s.rmse_days:>11.1f}  {s.mae_days:>10.1f}")

# ── 3. Grid Search ────────────────────────────────────────────────────────
lines.append('')
lines.append('[ 3 ] HYPERPARAMETER TUNING — GRID SEARCH RESULTS')
lines.append(sep2)
lines.append(f'  Best cross validation RMSE (log scale): {-grid_search.best_score_:.4f}')
lines.append('  Best parameters:')
for k, v in grid_search.best_params_.items():
    lines.append(f'    {k:<30} {v}')
lines.append('')
lines.append('  Full grid results (mean_test_score = negative RMSE):')
cv_res = pd.DataFrame(grid_search.cv_results_)
cols_show = ['param_rf__n_estimators','param_rf__max_depth','param_rf__min_samples_leaf',
             'param_rf__max_features','mean_test_score','std_test_score','rank_test_score']
cv_res_show = cv_res[cols_show].sort_values('rank_test_score')
cv_res_show.columns = ['n_est','max_depth','min_leaf','max_feat','mean_neg_rmse','std','rank']
lines.append(f'  {cv_res_show.to_string(index=False)}')

# ── 4. Tuned Cross Validation ─────────────────────────────────────────────
lines.append('')
lines.append('[ 4 ] TUNED RANDOM FOREST — 5-FOLD CROSS VALIDATION')
lines.append(sep2)
lines.append(f"  {'Fold':>4}  {'R²':>7}  {'RMSE(log)':>10}  {'MAE(log)':>9}  {'RMSE(days)':>11}  {'MAE(days)':>10}")
lines.append('  ' + '-' * 60)
for _, row in tuned_df.iterrows():
    lines.append(f"  {int(row.fold):>4}  {row.r2:>7.4f}  {row.rmse_log:>10.4f}  {row.mae_log:>9.4f}  {row.rmse_days:>11.1f}  {row.mae_days:>10.1f}")
lines.append('  ' + '-' * 60)
m = tuned_df.drop('fold', axis=1).mean()
s = tuned_df.drop('fold', axis=1).std()
lines.append(f"  {'MEAN':>4}  {m.r2:>7.4f}  {m.rmse_log:>10.4f}  {m.mae_log:>9.4f}  {m.rmse_days:>11.1f}  {m.mae_days:>10.1f}")
lines.append(f"  {'STD':>4}  {s.r2:>7.4f}  {s.rmse_log:>10.4f}  {s.mae_log:>9.4f}  {s.rmse_days:>11.1f}  {s.mae_days:>10.1f}")

# ── 5. Out-of-Fold Aggregate Metrics ──────────────────────────────────────
lines.append('')
lines.append('[ 5 ] OUT-OF-FOLD AGGREGATE METRICS')
lines.append(sep2)
oof_r2       = r2_score(y.values, oof_preds)
oof_rmse     = np.sqrt(mean_squared_error(y.values, oof_preds))
oof_mae      = mean_absolute_error(y.values, oof_preds)
oof_mae_days = mean_absolute_error(np.expm1(y.values), np.expm1(oof_preds))
residuals    = y.values - oof_preds
lines.append(f'  OOF R²              : {oof_r2:.4f}')
lines.append(f'  OOF RMSE (log)      : {oof_rmse:.4f}')
lines.append(f'  OOF MAE  (log)      : {oof_mae:.4f}')
lines.append(f'  OOF MAE  (days)     : {oof_mae_days:.1f}')
lines.append(f'  Residual mean       : {residuals.mean():.4f}  (bias check; ideal = 0)')
lines.append(f'  Residual std        : {residuals.std():.4f}')
lines.append(f'  Residual skewness   : {pd.Series(residuals).skew():.4f}')
lines.append(f'  Residual kurtosis   : {pd.Series(residuals).kurt():.4f}')

# ── 6. Feature Importance ─────────────────────────────────────────────────
lines.append('')
lines.append('[ 6 ] FEATURE IMPORTANCE (full-dataset fit, Mean Decrease in Impurity)')
lines.append(sep2)
lines.append(f"  {'Feature':<32} {'Type':<14} {'Importance':>10}  Bar")
lines.append('  ' + '-' * 65)
for _, row in imp_df.sort_values('importance', ascending=False).iterrows():
    bar = '█' * int(row['importance'] * 100)
    lines.append(f"  {row['feature']:<32} {row['type']:<14} {row['importance']:>10.4f}  {bar}")

# ── 7. Sanity Check Tables ────────────────────────────────────────────────
lines.append('')
lines.append('[ 7 ] SANITY CHECK — MEDIAN PREDICTED vs ACTUAL DAYS')
lines.append(sep2)
lines.append('  By Permit Type:')
lines.append(f'  {pt.to_string()}')
lines.append('')
lines.append('  By Permit Class:')
lines.append(f'  {pc.to_string()}')

# ── 8. Example Predictions ────────────────────────────────────────────────
lines.append('')
lines.append('[ 8 ] EXAMPLE INFERENCE PREDICTIONS')
lines.append(sep2)
lines.append('  Without comment features (median-imputed):')
for ex in examples:
    result = predict_permit_review_time(**ex)
    label  = f"{ex['permittypedesc']} | {ex['permitclass']} | zone={ex['zone_family']} | cost=${ex.get('estprojectcost',0):,}"
    lines.append(f'  {label}')
    lines.append(f"    Predicted : {result['predicted_days']:.0f} days")
    if 'ci_low_days' in result:
        lines.append(f"    80% CI    : {result['ci_low_days']:.0f} – {result['ci_high_days']:.0f} days")
    lines.append('')
lines.append('  With comment features supplied:')
lines.append(f'  New | Single Family/Duplex | zone=SF | $500k | max_cycle=3 | has_structural=1 | has_eca=1')
lines.append(f"    Predicted : {enriched['predicted_days']:.0f} days")
if 'ci_low_days' in enriched:
    lines.append(f"    80% CI    : {enriched['ci_low_days']:.0f} – {enriched['ci_high_days']:.0f} days")

# ── 9. Saved Artifacts ────────────────────────────────────────────────────
lines.append('')
lines.append('[ 9 ] SAVED ARTIFACTS')
lines.append(sep2)
for a in [
    'ModelWeights_RF_Comments.joblib      — fitted pipeline (preprocessor + tuned random forest)',
    'cv_metrics_per_fold_comments.png     — per-fold R², RMSE, MAE bar charts',
    'oof_pred_vs_actual_comments.png      — out-of-fold predicted vs actual scatter',
    'residual_analysis_comments.png       — residual histogram and residuals vs predicted',
    'feature_importance_comments.png      — Mean Decrease in Impurity feature importance bar chart',
    'DiagnosticReport_RF_Comments.txt     — this report',
]:
    lines.append(f'  {a}')

lines.append('')
lines.append(sep)
lines.append('  END OF REPORT')
lines.append(sep)

report_str = '\n'.join(lines)
print(report_str)

report_path = f'{OUTPUT_DIR}DiagnosticReport_RF_Comments.txt'
with open(report_path, 'w') as f:
    f.write(report_str)
print(f'\nReport saved: {report_path}')